In [ ]:
import pandas as pd
import numpy as np
import h3
import matplotlib.pyplot as plt
import seaborn as sns
import holidays
from pathlib import Path
from sklearn.svm import SVR, LinearSVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
print("Loading final grid dataset...")
final_grid = pd.read_parquet("../data/final_grid.parquet")
print(f"Final grid loaded. Shape: {final_grid.shape}")

final_grid.head()

Loading final grid dataset...
Final grid loaded. Shape: (9272950, 15)


,hour,Total_Trip_Start,Unique Taxis,Avg Trip Seconds,Avg Trip Miles,Avg Fare,Most Common Company,Company Count,Pickup Community Area,Dropoff Community Area,Total_Trip_End,poi_count,count_poi_types,poi_type_list,poi_list
0,2024-01-01,2.0,2.0,425.3300,2.400000,1812.500000,City Service,2.0,28.0,8.0,1.0,72.0,28.0,"school, bicycle_rental, bicycle_rental, bar, a...","Chicago Institute of Technology, Ashland Ave &..."
1,2024-01-01,7.0,7.0,1.7400,17.370000,5821.428571,Flash Cab,6.0,76.0,32.0,0.0,234.0,40.0,"cafe, newsagent, books, atm, convenience, fast...","Starbucks, Hudson News, Barbara's Bookstore, C..."
2,2024-01-01,8.0,8.0,440.6800,0.587500,1168.500000,Chicago Independents,6.0,8.0,8.0,5.0,316.0,57.0,"pub, restaurant, copyshop, bar, restaurant, un...","Midtown Kitchen & Bar, Nick's Fishmarket Grill..."
3,2024-01-01,1.0,1.0,2.8790,3.290000,2175.000000,Flash Cab,1.0,28.0,8.0,0.0,264.0,53.0,"ferry_terminal, university, music_school, gift...","Union Station/Willis Tower, Chicago-Kent Colle..."
4,2024-01-01,15.0,15.0,294.7794,1.155333,1502.666667,Chicago Independents,7.0,8.0,8.0,7.0,406.0,64.0,"college, art, restaurant, fast_food, hotel, ho...","Loyola Law School, R.H. Love Galleries, Jake M..."


In [ ]:
# DEFINE TARGET AND FEATURES

target = ''
features = [col for col in final_grid.columns if col != target]

In [ ]:
# Determine splitting date
unique_dates = pd.Series(final_grid['hour'].dt.date.unique()).sort_values()
split_idx = int(len(unique_dates) * 0.8)
split_date = unique_dates.iloc[split_idx]
print(f"Temporal Split Date: {split_date}")

train_mask = final_grid['hour'].dt.date < split_date
test_mask = final_grid['hour'].dt.date >= split_date

df_train = final_grid[train_mask]
df_test = final_grid[test_mask]
print(f"Full Train rows: {len(df_train)}, Test rows: {len(df_test)}")

# SVR Sample training set to prevent performance bottleneck
sample_size = 10000
df_train_sample = df_train.sample(n=sample_size, random_state=42)
print(f"Sub-sampled SVR Train rows: {len(df_train_sample)}")

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(df_train_sample[features])
y_train = df_train_sample['Total_Trip_Start'].values

X_test = scaler.transform(df_test[features])
y_test = df_test['Total_Trip_Start'].values

In [ ]:
KERNELS = ['linear','poly','rbf']
models_results = {}

for kernel in KERNELS:
    print(f"Training SVR with {kernel} kernel...")
    svr = SVR(kernel=kernel)
    svr.fit(X_train, y_train)
    print(f"Predicting with SVR ({kernel})...")
    y_pred = svr.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"SVR ({kernel}) - MAE: {mae:.2f}, MSE: {mse:.2f}, R²: {r2:.4f}")
    models_results[f'SVR ({kernel})'] = y_pred

best_kernel = max(models_results, key=lambda k: r2_score(y_test, models_results[k]))
print(f"Best SVR kernel based on R²: {best_kernel}")

In [ ]:
# Grid Search Optimization on best kernel
print("Running GridSearchCV on " + best_kernel + " SVR on a smaller sample (5000 rows)...")
X_grid_sample = X_train[:5000]
y_grid_sample = y_train[:5000]

param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 0.01, 0.1]
}

grid_search = GridSearchCV(SVR(kernel=best_kernel, max_iter=10000), param_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_search.fit(X_grid_sample, y_grid_sample)
best_svr = grid_search.best_estimator_
print(f"Best Parameters: {grid_search.best_params_}")

y_pred_opt = best_svr.predict(X_test)
models_results['Optimized ' + best_kernel] = y_pred_opt

In [ ]:

evaluation_metrics = []
for name, preds in models_results.items():
    # SVR predicts continuous values, clip predicted demand at 0
    clipped_preds = np.clip(preds, 0, None)
    mae = mean_absolute_error(y_test, clipped_preds)
    rmse = np.sqrt(mean_squared_error(y_test, clipped_preds))
    r2 = r2_score(y_test, clipped_preds)
    evaluation_metrics.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    })

df_metrics = pd.DataFrame(evaluation_metrics)
print("\n--- Model Performance Comparison ---")
print(df_metrics.to_string(index=False))

# Plot actual vs predicted values for RBF SVR
plt.figure(figsize=(10, 6))
plt.scatter(y_test, np.clip(y_pred_opt, 0, None), alpha=0.3, color='royalblue')
plt.plot([0, y_test.max()], [0, y_test.max()], 'r--', lw=2)
plt.title("Optimized RBF SVR: Actual vs. Predicted Demand")
plt.xlabel("Actual Trip Count")
plt.ylabel("Predicted Trip Count")
plt.tight_layout()
plt.show()

In [ ]:
# TODO: Add Spatial and Temporal Resolution Analysis
# TODO: Spatial Resolution with Census Tracts
# TODO: Answering the "Outskirts Rainy Sunday" Scenario